# Steering Vectors for SANA Diversity

This notebook computes steering vectors for SANA and generates diverse images.

**Requirements**: T4 GPU (Colab free tier) or better.

In [ ]:
!pip install -q diffusers transformers accelerate torch
!pip install -q open_clip_torch vendi-score image-reward sentencepiece

In [ ]:
import os
import random
import pickle
import numpy as np
from collections import defaultdict
from PIL import Image
import matplotlib.pyplot as plt

import torch
from diffusers import SanaPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load SANA Pipeline

In [ ]:
MODEL_ID = 'Efficient-Large-Model/Sana_600M_512px_BF16_diffusers'

pipe = SanaPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
)
pipe.to(device)

num_layers = len(pipe.transformer.transformer_blocks)
print(f'SANA loaded: {num_layers} transformer blocks')

## 2. Controller (adapted for SANA)

In [ ]:
import abc
from typing import Optional, Dict, Any


class VectorControl(abc.ABC):
    def __init__(self):
        self.cur_step = 0
        self.num_att_layers = -1
        self.cur_att_layer = 0

    def reset(self):
        self.cur_step = 0
        self.cur_att_layer = 0

    def between_steps(self):
        return

    @abc.abstractmethod
    def forward(self, vector, layer_idx: int):
        raise NotImplementedError

    def __call__(self, vector, layer_idx: int):
        vector = self.forward(vector, layer_idx)
        self.cur_att_layer += 1
        if self.cur_att_layer == self.num_att_layers:
            self.cur_att_layer = 0
            self.between_steps()
            self.cur_step += 1
        return vector


class SanaVectorStore(VectorControl):
    def __init__(self, steering_vectors=None, steer=True,
                 alpha=10, beta=2, steer_back=False,
                 active_layers=None, timestep_scaling=False,
                 total_steps=20, device='cpu'):
        super().__init__()
        self.step_store = {"layers": []}
        self.vector_store = defaultdict(dict)
        self.steering_vectors = steering_vectors
        self.steer = steer
        self.alpha = alpha
        self.beta = beta
        self.steer_back = steer_back
        self.device = device
        self.active_layers = active_layers
        self.timestep_scaling = timestep_scaling
        self.total_steps = total_steps

    def reset(self):
        super().reset()
        self.step_store = {"layers": []}
        self.vector_store = defaultdict(dict)

    def _get_alpha(self):
        if self.timestep_scaling:
            decay = 1.0 - (self.cur_step / max(self.total_steps, 1))
            return self.alpha * max(decay, 0.1)
        return self.alpha

    def forward(self, vector, layer_idx: int):
        if self.steer and self.steering_vectors is not None:
            if self.active_layers is None or layer_idx in self.active_layers:
                keys = sorted(self.steering_vectors.keys())
                if len(keys) == 1:
                    num_steer = keys[0]
                else:
                    num_steer = min(self.cur_step, keys[-1])

                sv = self.steering_vectors[num_steer]["layers"][layer_idx]
                sv = torch.tensor(sv).to(self.device).view(1, 1, -1)
                norm = torch.norm(vector, dim=2, keepdim=True)

                if self.steer_back:
                    sim = torch.tensordot(vector, sv, dims=([2], [2])).view(
                        vector.size()[0], vector.size()[1], 1)
                    sim = torch.where(sim > 0, sim, 0)
                    vector = vector - (self.beta * sim) * sv.expand(1, vector.size()[1], -1)
                else:
                    alpha = self._get_alpha()
                    vector = vector + alpha * sv.expand(1, vector.size()[1], -1)

                vector = vector / torch.norm(vector, dim=2, keepdim=True) * norm

        self.step_store["layers"].append(
            vector.data.cpu().numpy()[len(vector)//2:].mean(axis=0).mean(axis=0)
        )
        return vector

    def between_steps(self):
        self.vector_store[self.cur_step] = self.step_store
        self.step_store = {"layers": []}


def register_vector_control_sana(model, controller, hook_point="cross_attn"):
    def block_forward(block, layer_idx):
        def forward(
            hidden_states: torch.Tensor,
            attention_mask=None,
            encoder_hidden_states=None,
            encoder_attention_mask=None,
            timestep=None,
            height: int = None,
            width: int = None,
        ) -> torch.Tensor:
            batch_size = hidden_states.shape[0]
            shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
                block.scale_shift_table[None] + timestep.reshape(batch_size, 6, -1)
            ).chunk(6, dim=1)

            norm_hidden_states = block.norm1(hidden_states)
            norm_hidden_states = norm_hidden_states * (1 + scale_msa) + shift_msa
            norm_hidden_states = norm_hidden_states.to(hidden_states.dtype)
            attn_output = block.attn1(norm_hidden_states)
            if hook_point == "self_attn":
                attn_output = controller(attn_output, layer_idx)
            hidden_states = hidden_states + gate_msa * attn_output

            if block.attn2 is not None:
                attn_output = block.attn2(
                    hidden_states,
                    encoder_hidden_states=encoder_hidden_states,
                    attention_mask=encoder_attention_mask,
                )
                if hook_point == "cross_attn":
                    attn_output = controller(attn_output, layer_idx)
                hidden_states = attn_output + hidden_states

            if hook_point == "residual":
                hidden_states = controller(hidden_states, layer_idx)

            norm_hidden_states = block.norm2(hidden_states)
            norm_hidden_states = norm_hidden_states * (1 + scale_mlp) + shift_mlp
            norm_hidden_states = norm_hidden_states.unflatten(1, (height, width)).permute(0, 3, 1, 2)
            ff_output = block.ff(norm_hidden_states)
            ff_output = ff_output.flatten(2, 3).permute(0, 2, 1)
            hidden_states = hidden_states + gate_mlp * ff_output
            return hidden_states

        return forward

    count = 0
    for i, block in enumerate(model.transformer_blocks):
        block.forward = block_forward(block, i)
        count += 1
    controller.num_att_layers = count
    print(f"Registered controller for {count} blocks (hook: {hook_point})")

## 3. Compute Steering Vectors

Two averaging modes are available:
- **Multi-concept** (default): many ImageNet concepts, one prompt each → diversity bank
- **Per-concept** (like SD): one concept, many prompts averaged → single SV

In [ ]:
# === Choose averaging mode ===
AVERAGING_MODE = "multi_concept"  # "multi_concept" or "per_concept"

# --- Multi-concept settings ---
# ImageNet classes (subset)
CONCEPTS = [
    "tench", "goldfish", "shark", "hen", "ostrich", "brambling", "goldfinch",
    "house finch", "junco", "indigo bunting", "robin", "bulbul", "jay", "magpie",
    "water ouzel", "kite", "bald eagle", "vulture", "agama", "chameleon",
    "iguana", "alligator lizard", "triceratops", "scorpion", "tarantula",
    "centipede", "peacock", "flamingo", "pelican", "king penguin",
    "hummingbird", "toucan", "drake", "goose", "black swan",
    "dalmatian", "poodle", "collie", "golden retriever", "labrador",
    "persian cat", "siamese cat", "tabby cat", "tiger", "lion",
    "panda", "polar bear", "gorilla", "chimpanzee", "zebra",
]

# --- Per-concept settings (like SD) ---
CONCEPT_POS = "anime"
CONCEPT_NEG = None
PROMPT_MODE = "style"  # "style", "concrete", or "human-related"

NUM_STEPS = 20
HOOK_POINT = "cross_attn"

# Build prompt pairs
if AVERAGING_MODE == "multi_concept":
    prompts_pos = [f"a photo of {cls}" for cls in CONCEPTS]
    prompts_neg = ["a photo"] * len(CONCEPTS)
    print(f"Multi-concept mode: {len(CONCEPTS)} concepts")
elif AVERAGING_MODE == "per_concept":
    # Reuse SD-style prompt templates
    def get_prompts_style(concepts, concept_pos, concept_neg=None):
        pos, neg = [], []
        for cls in concepts:
            pos.append(f"{cls}, {concept_pos} style")
            neg.append(f"{cls}, {concept_neg} style" if concept_neg else cls)
        return pos, neg

    def get_prompts_concrete(concepts, concept_pos, concept_neg=None):
        pos, neg = [], []
        for cls in concepts:
            pos.append(f"{cls} with {concept_pos}")
            neg.append(f"{cls} with {concept_neg}" if concept_neg else cls)
        return pos, neg

    def get_prompts_human_related(concept_pos, concept_neg=None):
        B = ['a girl', 'two men', 'a man', 'a woman', 'an old man', 'a boy', 'boys', 'group of people']
        C = ['on a beach', 'zoomed in', 'talking', 'dancing on the street', 'playing guitar',
             'enjoying nature', 'smiling', 'in futuristic spaceship', 'with kittens',
             'in a strange pose', 'realism', 'colorful background', '']
        pos, neg = [], []
        for b in B:
            for c in C:
                pos.append(f"{b} {c}, {concept_pos}")
                neg.append(f"{b} {c}, {concept_neg}" if concept_neg else f"{b} {c}")
        return pos, neg

    if PROMPT_MODE == "style":
        prompts_pos, prompts_neg = get_prompts_style(CONCEPTS, CONCEPT_POS, CONCEPT_NEG)
    elif PROMPT_MODE == "concrete":
        prompts_pos, prompts_neg = get_prompts_concrete(CONCEPTS, CONCEPT_POS, CONCEPT_NEG)
    elif PROMPT_MODE == "human-related":
        prompts_pos, prompts_neg = get_prompts_human_related(CONCEPT_POS, CONCEPT_NEG)
    print(f"Per-concept mode: '{CONCEPT_POS}' vs '{CONCEPT_NEG}', {len(prompts_pos)} prompt pairs ({PROMPT_MODE})")

In [ ]:
pos_vectors = []
neg_vectors = []
seed = 0

for i, (prompt_pos, prompt_neg) in enumerate(zip(prompts_pos, prompts_neg)):
    print(f"[{i+1}/{len(prompts_pos)}] pos=\"{prompt_pos}\", neg=\"{prompt_neg}\"")

    # Positive
    ctrl = SanaVectorStore(device=device)
    ctrl.steer = False
    register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
    _ = pipe(prompt=prompt_pos, num_inference_steps=NUM_STEPS,
             generator=torch.Generator(device=device).manual_seed(seed))
    pos_vectors.append(ctrl.vector_store)

    # Negative
    ctrl = SanaVectorStore(device=device)
    ctrl.steer = False
    register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
    _ = pipe(prompt=prompt_neg, num_inference_steps=NUM_STEPS,
             generator=torch.Generator(device=device).manual_seed(seed))
    neg_vectors.append(ctrl.vector_store)

print("Done collecting activations!")

In [ ]:
num_block_layers = len(pos_vectors[0][0]["layers"])

# Compute averaged steering vector (used by both modes)
steering_vectors_avg = {}
for step in range(NUM_STEPS):
    steering_vectors_avg[step] = {"layers": []}
    for layer in range(num_block_layers):
        pv = [pos_vectors[i][step]["layers"][layer] for i in range(len(pos_vectors))]
        nv = [neg_vectors[i][step]["layers"][layer] for i in range(len(neg_vectors))]
        sv = np.mean(pv, axis=0) - np.mean(nv, axis=0)
        sv = sv / np.linalg.norm(sv)
        steering_vectors_avg[step]["layers"].append(sv)

os.makedirs('steering_vectors_sana', exist_ok=True)

if AVERAGING_MODE == "per_concept":
    # Single concept averaged across prompts (like SD)
    save_path = f'steering_vectors_sana/sana_{HOOK_POINT}_{CONCEPT_POS}_{CONCEPT_NEG}.pickle'
    with open(save_path, 'wb') as f:
        pickle.dump(steering_vectors_avg, f)
    single_sv = steering_vectors_avg
    concept_bank = None
    print(f"Saved per-concept averaged SV to {save_path}")

elif AVERAGING_MODE == "multi_concept":
    with open(f'steering_vectors_sana/sana_{HOOK_POINT}_avg.pickle', 'wb') as f:
        pickle.dump(steering_vectors_avg, f)

    # Also build per-concept bank
    concept_bank = {}
    for idx, concept in enumerate(CONCEPTS):
        concept_sv = {}
        for step in range(NUM_STEPS):
            concept_sv[step] = {"layers": []}
            for layer in range(num_block_layers):
                pv = pos_vectors[idx][step]["layers"][layer]
                nv = neg_vectors[idx][step]["layers"][layer]
                sv = pv - nv
                sv = sv / np.linalg.norm(sv)
                concept_sv[step]["layers"].append(sv)
        concept_bank[concept] = concept_sv

    with open('steering_vectors_sana/concept_bank.pickle', 'wb') as f:
        pickle.dump(concept_bank, f)
    single_sv = None
    print(f"Saved bank with {len(concept_bank)} concepts")

## 4. Generate: Baseline vs Steered

In [ ]:
TEST_PROMPTS = [
    "a photo of a girl",
    "a cat sitting on a windowsill",
    "a mountain landscape at sunset",
    "a red sports car on a highway",
    "a bouquet of flowers in a vase",
    "an astronaut floating in space",
    "a cozy coffee shop interior",
    "a dog playing in the park",
    "a futuristic city skyline",
    "a plate of sushi on a wooden table",
]

NUM_SEEDS = 10
ALPHA = 10

if concept_bank is not None:
    concept_names = list(concept_bank.keys())

In [ ]:
# Generate baseline
baseline_images = {}  # prompt_idx -> [images]

for pi, prompt in enumerate(TEST_PROMPTS):
    baseline_images[pi] = []
    for seed in range(NUM_SEEDS):
        ctrl = SanaVectorStore(device=device)
        ctrl.steer = False
        register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
        img = pipe(prompt=prompt, num_inference_steps=NUM_STEPS,
                   generator=torch.Generator(device=device).manual_seed(seed)).images[0]
        baseline_images[pi].append(img)
    print(f"Baseline: prompt {pi} done")

print("Baseline generation complete!")

In [ ]:
# Generate steered (strategy: all_layers, cross_attn)
steered_images = {}  # prompt_idx -> [images]

for pi, prompt in enumerate(TEST_PROMPTS):
    steered_images[pi] = []
    for seed in range(NUM_SEEDS):
        if single_sv is not None:
            # Per-concept mode: use the single averaged SV
            sv = single_sv
        else:
            # Multi-concept mode: pick random concept from bank
            concept = random.choice(concept_names)
            sv = concept_bank[concept]

        ctrl = SanaVectorStore(
            steering_vectors=sv, steer=True,
            alpha=ALPHA, device=device,
            total_steps=NUM_STEPS,
        )
        register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
        img = pipe(prompt=prompt, num_inference_steps=NUM_STEPS,
                   generator=torch.Generator(device=device).manual_seed(seed)).images[0]
        steered_images[pi].append(img)
    print(f"Steered: prompt {pi} done")

print("Steered generation complete!")

## 5. Visual Comparison

In [ ]:
# Show first 5 seeds for each of first 3 prompts
for pi in range(3):
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(f'Prompt: "{TEST_PROMPTS[pi]}"', fontsize=14)

    for s in range(5):
        axes[0, s].imshow(baseline_images[pi][s])
        axes[0, s].set_title(f'Baseline seed={s}')
        axes[0, s].axis('off')

        axes[1, s].imshow(steered_images[pi][s])
        axes[1, s].set_title(f'Steered seed={s}')
        axes[1, s].axis('off')

    plt.tight_layout()
    plt.show()

## 6. Evaluate

In [ ]:
import open_clip
import torch.nn.functional as F

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

def get_image_features(images):
    feats = []
    for img in images:
        t = clip_preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            f = clip_model.encode_image(t)
            f = F.normalize(f, dim=-1)
        feats.append(f)
    return torch.cat(feats)

def clip_score(images, prompt):
    img_f = get_image_features(images)
    tokens = clip_tokenizer([prompt]).to(device)
    with torch.no_grad():
        txt_f = clip_model.encode_text(tokens)
        txt_f = F.normalize(txt_f, dim=-1)
    return (img_f @ txt_f.T).squeeze(-1).cpu().numpy()

def mps(images):
    if len(images) < 2:
        return 1.0
    feats = get_image_features(images)
    sim = feats @ feats.T
    n = len(images)
    mask = torch.triu(torch.ones(n, n, device=device), diagonal=1).bool()
    return sim[mask].mean().item()

def vendi_score(images):
    from vendi_score import vendi
    feats = get_image_features(images)
    sim = (feats @ feats.T).cpu().numpy()
    return vendi.score_K(sim)

In [ ]:
print(f"{'Prompt':<45} | {'CLIPScore B':>11} | {'CLIPScore S':>11} | {'MPS B':>7} | {'MPS S':>7} | {'Vendi B':>8} | {'Vendi S':>8}")
print("-" * 115)

all_cs_b, all_cs_s = [], []
all_mps_b, all_mps_s = [], []
all_vs_b, all_vs_s = [], []

for pi, prompt in enumerate(TEST_PROMPTS):
    cs_b = clip_score(baseline_images[pi], prompt).mean()
    cs_s = clip_score(steered_images[pi], prompt).mean()
    mps_b = mps(baseline_images[pi])
    mps_s = mps(steered_images[pi])
    vs_b = vendi_score(baseline_images[pi])
    vs_s = vendi_score(steered_images[pi])

    all_cs_b.append(cs_b); all_cs_s.append(cs_s)
    all_mps_b.append(mps_b); all_mps_s.append(mps_s)
    all_vs_b.append(vs_b); all_vs_s.append(vs_s)

    print(f"{prompt:<45} | {cs_b:>11.4f} | {cs_s:>11.4f} | {mps_b:>7.4f} | {mps_s:>7.4f} | {vs_b:>8.4f} | {vs_s:>8.4f}")

print("-" * 115)
print(f"{'AVERAGE':<45} | {np.mean(all_cs_b):>11.4f} | {np.mean(all_cs_s):>11.4f} | {np.mean(all_mps_b):>7.4f} | {np.mean(all_mps_s):>7.4f} | {np.mean(all_vs_b):>8.4f} | {np.mean(all_vs_s):>8.4f}")
print(f"\nB = Baseline, S = Steered")
print(f"CLIPScore: higher = better quality")
print(f"MPS: lower = more diverse")
print(f"Vendi: higher = more diverse")